## Base Overlay

In [ ]:
import pynq
import numpy as np
import asyncio
import time
from base import BaseOverlay

# Make sure "base.bit" and "base.hwh" are in the same directory
print("Loading Base Overlay...")
ol = BaseOverlay("base.bit")
print("Base Overlay loaded successfully!\n")

print("=== Base Overlay Metadata ===")
print(f"Bitstream Path : {ol.bitfile_name}")
print(f"Timestamp      : {ol.timestamp}")

print("\n=== Available IPs (ip_dict) ===")
for ip_name in ol.ip_dict.keys():
    print(f"  - {ip_name}")

print("\n=== Available Interrupts (interrupt_pins) ===")
for intr_name in ol.interrupt_pins.keys():
    print(f"  - {intr_name}")

print("\n=== Driver Binding Verification ===")
# Verify that the custom AxiTimer driver was automatically bound
print(f"Timer IP Class : {type(ol.timer)}")

## GPIO Test

In [ ]:
# ==========================================
# 1. Test LED
# ==========================================
print("--- Testing LED ---")
ol.write_led(1)
print("LED is ON. Check your board.")
time.sleep(1)
ol.write_led(0)
print("LED is OFF.")

# ==========================================
# 2. Test Binary Counter (32-bit)
# ==========================================
print("\n--- Testing Binary Counter ---")
count1 = ol.read_binary_counter()
time.sleep(0.5)
count2 = ol.read_binary_counter()
print(f"Counter Read 1: {count1}")
print(f"Counter Read 2: {count2}")
if count2 is not None and count1 is not None and count2 > count1:
    print("Binary counter is ticking correctly!")

# ==========================================
# 3. Test User IO (4-bit)
# ==========================================
print("\n--- Testing User IO (4-bit) ---")
# Writing a 4-bit value (e.g., 0xA = 1010 in binary)
test_val_user = 0xA
ol.write_userio(test_val_user)
print(f"Wrote {hex(test_val_user)} to User IO Output (Ch2).")
# Note: Unless you physically loop back User IO output pins to input pins,
# reading from Ch1 will likely just return floating/default states.
user_in = ol.read_userio()
print(f"Read {hex(user_in) if user_in is not None else 'None'} from User IO Input (Ch1).")

# ==========================================
# 4. Test Fiber IO (2-bit)
# ==========================================
print("\n--- Testing Fiber IO (2-bit) ---")
# Writing a 2-bit value (e.g., 0x3 = 11 in binary)
test_val_fiber = 0x3
ol.write_fiberio(test_val_fiber)
print(f"Wrote {hex(test_val_fiber)} to Fiber IO Output (Ch2).")
fiber_in = ol.read_fiberio()
print(f"Read {hex(fiber_in) if fiber_in is not None else 'None'} from Fiber IO Input (Ch1).")

## AXI Interrupt Controller

### AXI Timer Interrupt

In [ ]:
async def test_timer_interrupt():
    print("--- Testing AXI Timer & Interrupts ---")
    
    # 1. Check current UIO interrupt status
    print("\n[Pre-Test] UIO Interrupt Counters:")
    ol.check_interrupts(filter_keyword="uio")
    
    # 2. Start timer for 2 seconds
    delay = 2.0
    print(f"\nStarting AXI Timer for {delay} seconds...")
    ol.timer.start(delay_sec=delay)
    
    # 3. Wait for the hardware interrupt asynchronously
    start_time = time.time()
    await ol.timer.wait_async()
    end_time = time.time()
    
    # 4. Clear the interrupt flag
    ol.timer.clear_interrupt()
    
    print(f"Timer Interrupt caught! Elapsed time: {end_time - start_time:.2f} seconds.")
    
    # 5. Check UIO interrupt status again (count should increment)
    print("\n[Post-Test] UIO Interrupt Counters:")
    ol.check_interrupts(filter_keyword="uio")

# Execute the async function
await test_timer_interrupt()

### AXI GPIO Interrupt

In [ ]:
async def test_gpio_interrupt():
    print("--- Testing GPIO Triggered Interrupt 0 ---")
    
    # Create a background task to listen for the interrupt
    # This simulates a responsive system waiting for external events
    listen_task = asyncio.create_task(ol.wait_gpio_interrupt())
    
    print("Listening for GPIO interrupt on 'intr_concat/In0'...")
    await asyncio.sleep(1) # Simulate doing other work
    
    print("Triggering hardware interrupt via GPIO pulse...")
    ol.trigger_gpio_interrupt()
    
    # Wait for the listener task to confirm reception
    await listen_task
    print("GPIO Interrupt successfully captured!")

# Execute the async function
await test_gpio_interrupt()

## DMA

In [ ]:
async def test_dma_loopback():
    print("--- Testing DMA Async Loopback ---")
    
    print("\n[Pre-Test] CMA Memory Status:")
    ol.check_cma_info()
    
    # Allocate 10MB of contiguous memory for input and output
    # 2,500,000 uint32 elements = 10,000,000 bytes
    num_elements = 2500000
    print(f"\nAllocating {num_elements * 4 / (1024*1024):.2f} MB of CMA memory for buffers...")
    input_buffer = pynq.allocate(shape=(num_elements,), dtype=np.uint32)
    output_buffer = pynq.allocate(shape=(num_elements,), dtype=np.uint32)
    
    # Fill input buffer with sequential data
    input_buffer[:] = np.arange(num_elements, dtype=np.uint32)
    
    print("\n[Mid-Test] CMA Memory Status (Notice the drop in CmaFree):")
    ol.check_cma_info()
    
    # Perform asynchronous DMA transfer
    print("\nInitiating DMA transfer (MM2S -> S2MM loopback)...")
    start_time = time.time()
    success = await ol.dma_transfer_async(input_buffer, output_buffer)
    end_time = time.time()
    
    if success:
        print(f"DMA Transfer completed in {end_time - start_time:.4f} seconds.")
        # Verify data integrity
        if np.array_equal(input_buffer, output_buffer):
            print("SUCCESS: Output data perfectly matches input data!")
        else:
            print("ERROR: Data mismatch detected!")
    else:
        print("DMA Transfer failed.")
        
    # Free the allocated CMA memory
    print("\nFreeing buffers...")
    del input_buffer
    del output_buffer
    
    print("\n[Post-Test] CMA Memory Status (CmaFree should recover):")
    ol.check_cma_info()

# Execute the async function
await test_dma_loopback()